In [ ]:
import cv2
import pytesseract
import pandas as pd
import numpy as np
from PIL import Image
import re
from datetime import datetime
import json

class SalesReportExtractor:
    def __init__(self):
        # Day mapping for number corresponding to day
        self.day_mapping = {
            'monday': 1, 'tuesday': 2, 'wednesday': 3, 
            'thursday': 4, 'friday': 5, 'saturday': 6, 'sunday': 7,
            'mon' : 1, 'tue' : 2, 'wed' : 3, 
        }
        
        # Coordinate regions for different parts of the form (adjust based on your images)
        self.regions = {
            'date_area': (0, 0, 400, 100),  # Top left area where date is
            'main_table': (0, 150, 1200, 800),  # Main transaction table area
            'showroom_area': (400, 0, 800, 100)  # Showroom name area
        }
    
    def preprocess_image(self, image_path):
        """Preprocess image for better OCR results"""
        # Read image
        img = cv2.imread(image_path)
        
        # Convert to grayscale
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        
        # Apply dilation and erosion to remove noise
        kernel = np.ones((1,1), np.uint8)
        img = cv2.dilate(gray, kernel, iterations=1)
        img = cv2.erode(img, kernel, iterations=1)
        
        # Apply blur to smooth out the image
        img = cv2.medianBlur(img, 3)
        
        # Threshold the image
        _, img = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        
        return img
    
    def extract_date_and_day(self, image):
        """Extract date and day from the top of the image"""
        height, width = image.shape
        date_region = image[0:100, 0:400]  # Adjust coordinates as needed
        
        # OCR on date region
        date_text = pytesseract.image_to_string(date_region, config='--psm 6').strip()
        
        # Extract date using regex patterns
        date_patterns = [
            r'(\d{1,2}/\d{1,2}/\d{4})',  # DD/MM/YYYY
            r'(\d{1,2}\.\d{1,2}\.\d{4})',  # DD.MM.YYYY  
            r'(\d{1,2}-\d{1,2}-\d{4})',  # DD-MM-YYYY
            r'(\d{4}-\d{1,2}-\d{1,2})'   # YYYY-MM-DD
        ]
        
        date_match = None
        for pattern in date_patterns:
            match = re.search(pattern, date_text)
            if match:
                date_match = match.group(1)
                break
        
        # Extract day of week
        day_match = None
        day_pattern = r'(monday|tuesday|wednesday|thursday|friday|saturday|sunday)'
        day_search = re.search(day_pattern, date_text.lower())
        if day_search:
            day_match = day_search.group(1).capitalize()
        
        return date_match, day_match
    
    def extract_showroom_name(self, image):
        """Extract showroom name from header"""
        height, width = image.shape
        showroom_region = image[0:100, 400:800]
        
        showroom_text = pytesseract.image_to_string(showroom_region, config='--psm 6').strip()
        
        # Look for showroom patterns
        if 'nova' in showroom_text.lower() or 'tradehub' in showroom_text.lower():
            return "Nova TradeHub"
        elif 'ny-th' in showroom_text.lower():
            return "NY-TH"
        
        return "Nova TradeHub"  # Default fallback
    
    def extract_table_data(self, image):
        """Extract transaction data from the main table"""
        height, width = image.shape
        table_region = image[150:height-100, 0:width]  # Adjust as needed
        
        # Use pytesseract to get word-level data with bounding boxes
        data = pytesseract.image_to_data(table_region, output_type=pytesseract.Output.DICT)
        
        # Filter out low confidence text
        filtered_data = []
        for i in range(len(data['text'])):
            if int(data['conf'][i]) > 30:  # Confidence threshold
                filtered_data.append({
                    'text': data['text'][i].strip(),
                    'left': data['left'][i],
                    'top': data['top'][i],
                    'width': data['width'][i],
                    'height': data['height'][i]
                })
        
        return self.parse_table_structure(filtered_data)
    
    def parse_table_structure(self, ocr_data):
        """Parse the OCR data into structured transactions"""
        transactions = []
        
        # Group text by rows (similar y-coordinates)
        rows = {}
        for item in ocr_data:
            if item['text'] and len(item['text'].strip()) > 0:
                row_key = item['top'] // 20  # Group by approximate row
                if row_key not in rows:
                    rows[row_key] = []
                rows[row_key].append(item)
        
        # Sort rows by y-coordinate
        sorted_rows = sorted(rows.items())
        
        for row_y, row_items in sorted_rows:
            # Sort items in row by x-coordinate
            row_items.sort(key=lambda x: x['left'])
            
            # Extract transaction data
            transaction = self.extract_transaction_from_row(row_items)
            if transaction:
                transactions.append(transaction)
        
        return transactions
    
    def extract_transaction_from_row(self, row_items):
        """Extract transaction data from a single row"""
        row_text = [item['text'] for item in row_items if item['text'].strip()]
        
        if len(row_text) < 3:  # Skip rows with insufficient data
            return None
        
        transaction = {
            'name': None,
            'sales_order_no': None,
            'amount': None,
            'reason': None
        }
        
        # Look for patterns in the row
        for text in row_text:
            # Sales order number pattern
            if re.match(r'1-\d{6}', text):
                transaction['sales_order_no'] = text
            
            # Amount pattern (numbers with decimals)
            elif re.match(r'\d+\.?\d*', text) and len(text) > 2:
                # Could be amount
                try:
                    amount = float(text.replace(',', ''))
                    if amount > 10:  # Reasonable amount threshold
                        transaction['amount'] = amount
                except:
                    pass
            
            # Name pattern (short uppercase text)
            elif text.isupper() and len(text) <= 10 and text.isalpha():
                if text in ['KEN', 'TOM', 'JOHN', 'MICHAEL', 'CLAYTON', 'ZOEWE']:
                    transaction['name'] = text
            
            # Reason patterns
            elif text.lower() in ['walk-in', 'facebook', 'repeat', 'appointment', 'referral', 'sales up']:
                transaction['reason'] = text.title()
        
        # Only return if we have essential data
        if transaction['name'] and transaction['sales_order_no']:
            return transaction
        
        return None
    
    def process_image(self, image_path):
        """Main processing function"""
        # Preprocess image
        processed_img = self.preprocess_image(image_path)
        
        # Extract date and day
        date_str, day_str = self.extract_date_and_day(processed_img)
        
        # Get day number
        day_num = self.day_mapping.get(day_str.lower(), 0) if day_str else 0
        
        # Extract showroom
        showroom = self.extract_showroom_name(processed_img)
        
        # Extract transactions
        transactions = self.extract_table_data(processed_img)
        
        # Create structured data
        structured_data = []
        for transaction in transactions:
            if transaction:
                row = {
                    'Date': date_str,
                    'Day of the Week': day_str,
                    'Number corresponding to the day': day_num,
                    'Name': transaction['name'],
                    'Showroom': showroom,
                    'Sales Order No.': transaction['sales_order_no'],
                    'Amount': transaction['amount'],
                    'Reason': transaction['reason']
                }
                structured_data.append(row)
        
        return structured_data
    
    def process_multiple_images(self, image_paths, output_csv_path):
        """Process multiple images and save to CSV"""
        all_data = []
        
        for img_path in image_paths:
            print(f"Processing {img_path}...")
            try:
                data = self.process_image(img_path)
                all_data.extend(data)
                print(f"Extracted {len(data)} transactions")
            except Exception as e:
                print(f"Error processing {img_path}: {str(e)}")
        
        # Create DataFrame and save
        df = pd.DataFrame(all_data)
        df.to_csv(output_csv_path, index=False)
        print(f"Saved {len(all_data)} total transactions to {output_csv_path}")
        
        return df

# Usage example
if __name__ == "__main__":
    extractor = SalesReportExtractor()
    
    # Process single image
    # data = extractor.process_image("sales_report.jpg")
    # print(data)
    
    # Process multiple images
    image_files = [
        "report1.jpg",
        "report2.jpg", 
        "report3.jpg"
        # Add your image paths here
    ]
    
    df = extractor.process_multiple_images(image_files, "extracted_sales_data.csv")
    print(df.head())